# P22: Down syndrome fetal cortex Multiome — one-notebook workflow

**Canonical notebook only.** Local Jupyter + Google Colab.

| | |
|---|---|
| CELLxGENE dataset | `f16c25da-15bd-46a4-9a3f-17093f27a2f1` |
| GEO | `GSE305153` |
| Expected cells | 248,998 (cell-level scale) |
| Expected donors | 30 = 15 normal + 15 complete trisomy 21 (inferential scale) |
| H5AD | ~1.57 GB (not in Git) |

## Visible modes

1. **simulation** — synthetic wiring only (default safe)
2. **metadata_census** — live catalog + planned full-cohort contract; no H5AD
3. **approved_real_analysis** — full H5AD after dated Professor Fang approval

Gates **G0–G9**. Data-size stages **R1–R6**. Capped models ≠ full dataset.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")

# ---------------------------------------------------------------------------
# Visible mode controls (edit these; env flags also work)
# ---------------------------------------------------------------------------
NOTEBOOK_MODE = os.getenv("P22_NOTEBOOK_MODE", "simulation")  # simulation | metadata_census | approved_real_analysis
RUN_REAL_DATA = os.getenv("P22_RUN_REAL_DATA", "0") == "1"
RUN_MODEL = os.getenv("P22_RUN_MODEL", "0") == "1"
# Optional Colab override examples:
# NOTEBOOK_MODE = "metadata_census"
# NOTEBOOK_MODE = "approved_real_analysis"; RUN_REAL_DATA = True; RUN_MODEL = True

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install only missing deps; keep session NumPy.
    import importlib
    need = []
    for pkg, mod in [("anndata", "anndata"), ("torch", "torch"), ("scikit-learn", "sklearn")]:
        try:
            importlib.import_module(mod)
        except ImportError:
            need.append(pkg)
    if need:
        import subprocess as _sp
        _sp.check_call([sys.executable, "-m", "pip", "install", "-q", *need])
    PROJECT_ROOT = Path("/content/p22")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    DATA_ROOT = Path(os.getenv("P22_DATA_ROOT", "/content/p22_data"))
    OUTPUT_BASE = Path(os.getenv("P22_OUTPUT_ROOT", "/content/p22_output"))
    # Lightweight package path: prefer uploaded/checkout src if present
    for candidate in (Path("/content/P22"), Path("/content/niw-eb1a/P22"), Path.cwd()):
        if (candidate / "src" / "p22").is_dir():
            PROJECT_ROOT = candidate
            break
else:
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD
    for candidate in (CWD, *CWD.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "p22").is_dir():
            PROJECT_ROOT = candidate
            break
    DATA_ROOT = Path(os.getenv("P22_DATA_ROOT", PROJECT_ROOT / "data" / "real"))
    OUTPUT_BASE = Path(os.getenv("P22_OUTPUT_ROOT", PROJECT_ROOT / "reports" / "generated" / "verification" / "down_syndrome"))

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from p22 import load_approvals
from p22.data.adequacy import catalog_donor_frame, evaluate_donor_adequacy
from p22.data.catalog import (
    DEFAULT_COLLECTION_ID,
    DEFAULT_DATASET_ID,
    EXPECTED_CELLS,
    EXPECTED_DONORS,
    EXPECTED_H5AD_BYTES,
    run_catalog_preflight,
)
from p22.data.census import (
    census_blocked_placeholder,
    data_scale_section,
    download_h5ad,
    estimate_download_gb,
    planned_acquisition_from_catalog,
    run_backed_census,
    verify_local_h5ad,
)
from p22.data.group_splits import (
    aggregate_donor_probabilities,
    build_repeated_group_split_report,
)
from p22.data.resources import (
    PRIMARY_CAP,
    SENSITIVITY_CAPS,
    ResourceMeasurement,
    build_resource_report,
    decide_atac_branch,
)
from p22.eval.data_scale import empty_scale_board, make_scale
from p22.eval.estimand import freeze_estimand
from p22.eval.faithfulness import INTERVENTIONS, intervention_table, run_all_interventions
from p22.eval.gates import ONE_NOTEBOOK_CONTRACT, empty_board, make_gate
from p22.eval.modes import MODE_APPROVED_REAL, MODE_METADATA_CENSUS, MODE_SIMULATION, resolve_mode
from p22.eval.runtime import collect_runtime_report
from p22.eval.validation import build_validation_report
from p22.models.fusion import GatedFusionModel
from p22.models.named_baselines import (
    BaselineResult,
    baseline_table,
    blocked_baseline,
    chr21_dosage_baseline,
    covariate_logistic_baseline,
    majority_class_baseline,
    not_applicable_baseline,
    pseudobulk_rna_logistic,
)
from p22.reports.evidence_package import EvidencePackage
from p22.testing.synthetic import make_synthetic_multimodal

SEED = 20260728
MODEL_INIT_SEED = 0
DATASET_ID = DEFAULT_DATASET_ID
COLLECTION_ID = DEFAULT_COLLECTION_ID
SOURCE_NOTEBOOK = PROJECT_ROOT / "P22_down_syndrome_all_in_one.ipynb"

approvals = load_approvals(PROJECT_ROOT / "plan" / "approvals.json") if (PROJECT_ROOT / "plan" / "approvals.json").is_file() else load_approvals()
PROFESSOR_APPROVED = bool(approvals.get("approval_recorded")) and approvals.get("approval_state") == "approved"
mode = resolve_mode(
    NOTEBOOK_MODE,
    professor_approved=PROFESSOR_APPROVED,
    run_real_data=RUN_REAL_DATA,
    run_model=RUN_MODEL,
)
if (RUN_REAL_DATA or RUN_MODEL or NOTEBOOK_MODE == MODE_APPROVED_REAL) and not PROFESSOR_APPROVED:
    # Do not crash metadata/simulation; hard-stop only when user forces real flags without approval.
    if RUN_REAL_DATA or RUN_MODEL:
        raise RuntimeError("Approval gate: dated Professor Fang approval required before real data/model flags.")

RUN_STAMP = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_ROOT = OUTPUT_BASE / f"run_{RUN_STAMP}"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def safe_write_text(path: Path, text: str) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path

def notebook_hash() -> str:
    if not SOURCE_NOTEBOOK.is_file():
        return "unknown"
    return hashlib.sha256(SOURCE_NOTEBOOK.read_bytes()).hexdigest()

board = empty_board()
scale_board = empty_scale_board()
EVIDENCE_BUCKETS = {"verified_real": [], "synthetic": [], "metadata_only": [], "blocked_or_unknown": []}

print(ONE_NOTEBOOK_CONTRACT)
print({
    "in_colab": IN_COLAB,
    "mode": mode.to_dict(),
    "python": sys.version.split()[0],
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "estimated_h5ad_gb": estimate_download_gb(),
    "professor_approved": PROFESSOR_APPROVED,
})


## G0 — Runtime (local / Colab)


In [ ]:
runtime = collect_runtime_report(
    project_root=PROJECT_ROOT if not IN_COLAB else OUTPUT_ROOT,
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    source_notebook=SOURCE_NOTEBOOK if SOURCE_NOTEBOOK.is_file() else OUTPUT_ROOT / "notebook_placeholder.ipynb",
    min_disk_gb=2.0 if IN_COLAB else 5.0,
)
if IN_COLAB and not SOURCE_NOTEBOOK.is_file():
    # Colab upload may omit repo notebook path; do not block runtime on that alone.
    runtime_status = "PASS" if not runtime.blocking_problems else runtime.status
else:
    runtime_status = runtime.status
board.set(make_gate("G0", runtime_status, evidence=runtime.to_dict(), unknowns=runtime.open_questions,
                    notes="outputs under reports/generated or /content/p22_output; never overwrite source"))
safe_write_text(OUTPUT_ROOT / "local_runtime_report.json", json.dumps(runtime.to_dict(), indent=2) + "\n")
print("G0:", runtime_status)
print(json.dumps({"disk_free_gb": runtime.disk_free_gb, "device": runtime.device, "in_colab": IN_COLAB}, indent=2))


## G1 / R1 — Live catalog identity + acquisition contract

Estimated H5AD download shown before any download. Download only in approved real mode.


In [ ]:
catalog = run_catalog_preflight()
summary = catalog.summary or {}
acquisition = planned_acquisition_from_catalog(
    summary,
    collection_id=COLLECTION_ID,
    dataset_id=DATASET_ID,
    allow_download=mode.allow_h5ad_download,
)
print("Estimated H5AD download GB:", estimate_download_gb(acquisition.expected_bytes))
print(json.dumps({
    "G1_status": catalog.status,
    "cells": summary.get("cell_count"),
    "donors": summary.get("donor_count"),
    "conditions": summary.get("condition_counts_from_donor_suffix"),
    "h5ad_present": (summary.get("h5ad_asset") or {}).get("present"),
    "fragment_present": (summary.get("atac_fragment_asset") or {}).get("present"),
    "R1_status": acquisition.status,
    "expected_bytes": acquisition.expected_bytes,
}, indent=2))

board.set(make_gate("G1", catalog.status, evidence=catalog.to_dict(), unknowns=catalog.open_questions,
                    notes="same-nucleus pairing unknown until H5AD"))
scale_board.set(make_scale("R1", acquisition.status, evidence=acquisition.to_dict(),
                           unknowns=acquisition.open_questions,
                           notes="immutable acquisition record; checksum after download"))
safe_write_text(OUTPUT_ROOT / "preflight.json", json.dumps(catalog.to_dict(), indent=2) + "\n")
safe_write_text(OUTPUT_ROOT / "acquisition.json", json.dumps(acquisition.to_dict(), indent=2) + "\n")
EVIDENCE_BUCKETS["metadata_only"].append("G1 catalog preflight")
EVIDENCE_BUCKETS["blocked_or_unknown"].append("R1 checksum pending until approved download")


## G2 / R2 / R3 — Donor adequacy + full-cohort census

Full-cohort census uses every catalog row before any 256-cell cap. Without approved H5AD, census is BLOCKED (honest).


In [ ]:
donor_ids = list(summary.get("donor_ids") or [])
donor_frame = catalog_donor_frame(donor_ids) if donor_ids else pd.DataFrame(columns=["donor_id", "condition"])
adequacy = evaluate_donor_adequacy(donor_frame, pairing_known=False)

h5ad_path = DATA_ROOT / f"{DATASET_ID}.h5ad"
census = census_blocked_placeholder("H5AD not loaded in current mode")
local_verify = None

if mode.allow_h5ad_download:
    print("Downloading full H5AD (~%.2f GB). Not a tiny subset." % estimate_download_gb(acquisition.expected_bytes))
    h5ad_path = download_h5ad(h5ad_path, url=acquisition.h5ad_url, expected_bytes=acquisition.expected_bytes, allow=True)
    local_verify = verify_local_h5ad(h5ad_path, expected_bytes=acquisition.expected_bytes)
    print(json.dumps(local_verify, indent=2))
    if not local_verify["ok"]:
        census = census_blocked_placeholder(local_verify["reason"] or "H5AD verify failed")
    else:
        census = run_backed_census(h5ad_path)
        # Re-evaluate adequacy with real per-donor cell counts when available
        if census.donor_table:
            real_donors = (
                pd.DataFrame(census.donor_table)
                .groupby("donor_id", as_index=False)
                .agg(condition=("condition", "first"), n_cells=("n_cells", "sum"))
            )
            adequacy = evaluate_donor_adequacy(real_donors, pairing_known=census.peak_block_present is True)
else:
    print("Census skipped: mode does not allow H5AD download.")
    print("Catalog donors (all 30 identities):")
    print(donor_frame.to_string(index=False) if len(donor_frame) else "(none)")

board.set(make_gate("G2", adequacy.status, evidence=adequacy.to_dict(), unknowns=adequacy.open_questions,
                    notes="catalog adequacy until post-QC H5AD retained counts exist"))
scale_board.set(make_scale("R2", census.status, evidence=census.to_dict(), unknowns=census.open_questions,
                           notes="full-cohort census before model cap"))
r3_status = adequacy.status if census.status != "BLOCKED" else "BLOCKED"
scale_board.set(make_scale("R3", r3_status, evidence={"adequacy": adequacy.to_dict(), "census": census.to_dict()},
                           notes="post-QC adequacy uses retained values when H5AD present"))
print("G2:", adequacy.status, "R2:", census.status, "R3:", r3_status)
EVIDENCE_BUCKETS["metadata_only"].append("G2 catalog donor adequacy")
EVIDENCE_BUCKETS["blocked_or_unknown"].append("R2/R3 full-cohort H5AD census")


## G3 / R4 — Resources, caps, ATAC B1/B2

Layers: **full-cohort** (census/pseudobulk all retained) vs **cell-level model** (256 primary; 64/128 sensitivity; 512/all if resources permit).


In [ ]:
fragment_present = bool((summary.get("atac_fragment_asset") or {}).get("present"))
atac_branch = decide_atac_branch(
    peak_block_present=census.peak_block_present,
    fragment_asset_present=fragment_present,
    fragment_build_approved=False,
    resource_budget_ok=False,
)
measurements = [
    ResourceMeasurement(name="catalog_h5ad_size", cells_per_donor_cap=None,
                        input_bytes=acquisition.expected_bytes, disk_free_gb=runtime.disk_free_gb,
                        notes="catalog/planned size"),
    ResourceMeasurement(name="planned_primary_cap", cells_per_donor_cap=PRIMARY_CAP,
                        notes="cell-level model layer primary cap; not full dataset"),
    ResourceMeasurement(name="planned_sensitivity_caps", cells_per_donor_cap=None,
                        notes=f"sensitivity {list(SENSITIVITY_CAPS)}; optional 512/all_retained if resources permit"),
]
resource = build_resource_report(
    measurements=measurements,
    atac_branch=atac_branch,
    h5ad_available=census.n_cells_raw is not None,
    approval_present=PROFESSOR_APPROVED,
)
board.set(make_gate("G3", resource.status, evidence=resource.to_dict(), unknowns=resource.open_questions,
                    next_owner="Professor Fang" if not PROFESSOR_APPROVED else "researcher"))
scale_board.set(make_scale(
    "R4",
    "INCONCLUSIVE" if census.status == "BLOCKED" else resource.status,
    evidence={
        "full_cohort_layer": "census/donor summaries/pseudobulk over all retained cells",
        "cell_level_layer": {"primary": PRIMARY_CAP, "sensitivity": list(SENSITIVITY_CAPS)},
        "capped_is_not_full_dataset": True,
        "atac_branch": atac_branch.to_dict(),
    },
    notes="two layers must stay labeled separately",
))
print(json.dumps({"G3": resource.status, "atac_branch": atac_branch.branch, "multimodal_allowed": atac_branch.multimodal_claim_allowed}, indent=2))


## G4 — Frozen estimand (before predictions)


In [ ]:
n_control = int((adequacy.checks or {}).get("n_control_donors") or 15)
n_ds = int((adequacy.checks or {}).get("n_ds_donors") or 15)
estimand_report = freeze_estimand(n_control_donors=n_control, n_ds_donors=n_ds)
board.set(make_gate("G4", estimand_report.status, evidence=estimand_report.to_dict(),
                    unknowns=estimand_report.open_questions, notes="protocol frozen before predictions"))
assert estimand_report.estimand.margin >= 0.07
print("G4:", estimand_report.status, "margin=", estimand_report.estimand.margin)
print(json.dumps(estimand_report.estimand.to_dict()["validation_plan"], indent=2))


## G5 — Donor-held-out leakage control

Simulation mode always runs synthetic zero-overlap proof. Real-data G5 stays BLOCKED without approval + H5AD.


In [ ]:
synth = make_synthetic_multimodal(n_donors=30, cells_per_donor=24, n_features_a=32, n_features_b=32, n_classes=2, seed=SEED)
synth_labels = np.asarray([0 if int(str(d).split("_")[1]) < 15 else 1 for d in synth.donor_ids])
split_report = build_repeated_group_split_report(
    synth.donor_ids, synth_labels,
    n_repeats=estimand_report.estimand.split_repeats,
    n_folds=estimand_report.estimand.folds_per_repeat,
    base_seed=SEED,
)
assert split_report.donor_overlap_count == 0
g5_status = "BLOCKED" if not (PROFESSOR_APPROVED and mode.allow_model_fit) else split_report.status
board.set(make_gate("G5", g5_status, evidence={
    "synthetic_split_status": split_report.status,
    "n_folds": len(split_report.folds),
    "donor_overlap_count": split_report.donor_overlap_count,
    "real_data": "BLOCKED" if g5_status == "BLOCKED" else "eligible",
}, notes="synthetic wiring proof; real matrix evaluation approval-gated"))
print("G5:", g5_status, "synthetic_overlap=", split_report.donor_overlap_count)
EVIDENCE_BUCKETS["synthetic"].append("G5 StratifiedGroupKFold zero-overlap wiring")


## G6 — Named baselines (order fixed)

Pseudobulk uses **all cells in the synthetic/full-retained layer** for that baseline; cell-level RNA/ATAC/concat/gated use capped model layer.


In [ ]:
fold0 = split_report.folds[0]
train_donors, test_donors = fold0.train_donors, fold0.test_donors
y, donors = synth_labels, np.asarray(synth.donor_ids, dtype=object)
rna, atac = synth.view_a, synth.view_b
train_idx, test_idx = fold0.train_index, fold0.test_index

results = [
    majority_class_baseline(y, donors),
    chr21_dosage_baseline(None, y, donors, train_donors, test_donors),
    covariate_logistic_baseline(None, y, donors, train_donors, test_donors),
    # Full-retained-cell pseudobulk on this layer (all synth cells per donor — not a 256 subsample)
    pseudobulk_rna_logistic(rna, y, donors, train_donors, test_donors, n_features=16),
]

def donor_bal_acc(matrix, name):
    scaler = StandardScaler()
    x_train = scaler.fit_transform(matrix[train_idx])
    x_test = scaler.transform(matrix[test_idx])
    model = LogisticRegression(max_iter=400, class_weight="balanced", random_state=MODEL_INIT_SEED)
    model.fit(x_train, y[train_idx])
    proba = model.predict_proba(x_test)[:, 1]
    agg = aggregate_donor_probabilities(proba, donors[test_idx], threshold=0.5)
    truth = (pd.DataFrame({"donor_id": donors[test_idx], "label": y[test_idx]})
             .groupby("donor_id")["label"].agg(lambda s: int(s.mode().iloc[0])))
    merged = agg.merge(truth.rename("label"), on="donor_id")
    score = float(balanced_accuracy_score(merged["label"], merged["prediction"]))
    return BaselineResult(name=name, status="measured", donor_balanced_accuracy=score,
                          n_donors=int(merged.shape[0]), n_cells=int(test_idx.size),
                          detail={"layer": "cell_level_capped_or_synth", "cap_note": "not full cohort claim"})

results.append(donor_bal_acc(rna, "rna_only"))
if atac_branch.multimodal_claim_allowed:
    results.append(donor_bal_acc(atac, "atac_only"))
    results.append(donor_bal_acc(np.hstack([rna, atac]), "rna_atac_concat"))
    results.append(blocked_baseline("gated_fusion", "gated fusion blocked without approved real model run"))
else:
    reason = f"ATAC branch {atac_branch.branch}; multimodal claim not allowed"
    results.extend([
        not_applicable_baseline("atac_only", reason),
        not_applicable_baseline("rna_atac_concat", reason),
        not_applicable_baseline("gated_fusion", reason),
    ])

g6_status = "BLOCKED" if not (PROFESSOR_APPROVED and mode.allow_model_fit) else "PASS"
baseline_rows = baseline_table(results)
board.set(make_gate("G6", g6_status, evidence={"baselines": baseline_rows, "atac_branch": atac_branch.branch},
                    unknowns=("real-data baselines not run",), notes="synthetic baseline wiring; not disease evidence"))
print(pd.DataFrame(baseline_rows)[["name", "status", "donor_balanced_accuracy", "not_applicable"]].to_string(index=False))
print("G6:", g6_status)
EVIDENCE_BUCKETS["synthetic"].append("G6 named baseline wiring")


## G7 — Routing faithfulness (held-out only)

Routing signals ≠ explanations ≠ cross-attention.


In [ ]:
n_features = 16
rna_scaler = StandardScaler().fit(rna[train_idx][:, :n_features])
atac_scaler = StandardScaler().fit(atac[train_idx][:, :n_features])
train_rna = rna_scaler.transform(rna[train_idx][:, :n_features])
test_rna = rna_scaler.transform(rna[test_idx][:, :n_features])
train_atac = atac_scaler.transform(atac[train_idx][:, :n_features])
test_atac = atac_scaler.transform(atac[test_idx][:, :n_features])
torch.manual_seed(MODEL_INIT_SEED)
gated = GatedFusionModel(n_features_a=n_features, n_features_b=n_features, n_classes=2, embed_dim=16, hidden_dim=32)
opt = torch.optim.Adam(gated.parameters(), lr=1e-2)
xa = torch.tensor(train_rna, dtype=torch.float32)
xb = torch.tensor(train_atac, dtype=torch.float32)
yt = torch.tensor(y[train_idx], dtype=torch.int64)
for _ in range(20):
    opt.zero_grad()
    out = gated(xa, xb)
    torch.nn.functional.cross_entropy(out.logits, yt).backward()
    opt.step()
effects = run_all_interventions(
    gated,
    views={"view_a": test_rna, "view_b": test_atac},
    labels=y[test_idx],
    donor_ids=donors[test_idx],
    train_views={"view_a": train_rna, "view_b": train_atac},
    metric_name="balanced_accuracy",
    seed=SEED,
)
intervention_rows = intervention_table(effects)
g7_status = "BLOCKED" if not (PROFESSOR_APPROVED and mode.allow_model_fit) else "PASS"
board.set(make_gate("G7", g7_status, evidence={"interventions": intervention_rows},
                    notes="synthetic intervention plumbing; routing signals only"))
print(pd.DataFrame(intervention_rows).to_string(index=False))
print("G7:", g7_status, "n_interventions=", len(INTERVENTIONS))
EVIDENCE_BUCKETS["synthetic"].append("G7 routing interventions")


## G8 / R5 — External validation + donor-limited scale

`GSE280175` = RNA-only. Off-ramps O1/O2/O3.


In [ ]:
validation = build_validation_report(marker_set=None, findings_available=False, approval_present=PROFESSOR_APPROVED)
board.set(make_gate("G8", validation.status, evidence=validation.to_dict(), unknowns=validation.open_questions,
                    next_owner="Professor Fang", notes=validation.claim_boundary))
scale_board.set(make_scale(
    "R5",
    "INCONCLUSIVE",
    evidence={
        "cell_level_scale": EXPECTED_CELLS,
        "inferential_scale_donors": EXPECTED_DONORS,
        "statement_cells": f"Real full cohort target contains {EXPECTED_CELLS} cells and {EXPECTED_DONORS} donors.",
        "statement_donors": "Inferential uncertainty is donor-limited; donor resampling and external cohort evidence required.",
        "off_ramps": adequacy.off_ramp_options,
        "validation": validation.to_dict(),
    },
    notes="30 donors moderate; do not inflate via cell count",
))
print("G8:", validation.status)
print(json.dumps(validation.to_dict(), indent=2)[:1200])
EVIDENCE_BUCKETS["blocked_or_unknown"].append("G8 external validation")


## G9 / R6 — Evidence package


In [ ]:
def git_head() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
    except Exception:
        return "unknown"

data_scale = data_scale_section(
    acquisition, census,
    primary_cap=PRIMARY_CAP,
    sensitivity_caps=SENSITIVITY_CAPS,
)
scale_board.set(make_scale("R6", "PASS", evidence=data_scale, notes="numeric data-scale section written"))

package = EvidencePackage(
    run_dir=OUTPUT_ROOT / "runs" / f"{mode.mode}_{SEED}",
    manifest={
        "run_id": f"{mode.mode}_{SEED}",
        "commit": git_head(),
        "notebook_hash": notebook_hash(),
        "mode": mode.to_dict(),
        "dataset_id": DATASET_ID,
        "flags": {"run_real_data": RUN_REAL_DATA, "run_model": RUN_MODEL, "in_colab": IN_COLAB},
        "catalog": catalog.to_dict(),
        "acquisition": acquisition.to_dict(),
        "census": census.to_dict(),
        "adequacy": adequacy.to_dict(),
        "resource": resource.to_dict(),
        "estimand": estimand_report.to_dict(),
        "gates": board.to_dict(),
        "data_size_stages": scale_board.to_dict(),
        "data_scale": data_scale,
        "approval_state": approvals.get("approval_state"),
        "primary_cap": PRIMARY_CAP,
        "sensitivity_caps": list(SENSITIVITY_CAPS),
        "split_seeds_note": "split seed = base_seed + repeat; model_init_seed=0",
        "model_init_seed": MODEL_INIT_SEED,
        "evidence_buckets": EVIDENCE_BUCKETS,
        "limitations": [
            "Default/safe run is simulation or metadata; not disease evidence.",
            "Real H5AD/model blocked without dated Professor Fang approval.",
            "Capped cell-level models are not the full dataset.",
            "Inferential unit is donor (n≈30), not cell (n≈248998).",
            "GSE280175 is RNA-only validation, not multimodal.",
            "Routing weights are signals, not explanations or cross-attention.",
        ],
        "decision": "stop" if not PROFESSOR_APPROVED else "continue",
        "next_decision_owner": "Professor Fang",
        "next_decision": "record dated approval before approved_real_analysis",
    },
    metrics_rows=baseline_rows,
    intervention_rows=intervention_rows,
    validation_rows=[item.to_dict() | {"gate_status": validation.status} for item in validation.resources],
)
# Attach G9 before writing final summary files
board.set(make_gate("G9", "PASS", evidence={"run_dir": str(package.run_dir)}, notes="lean package path prepared"))
package.manifest["gates"] = board.to_dict()
paths = package.write()

gate_df = pd.DataFrame(board.as_rows())
scale_df = pd.DataFrame(scale_board.as_rows())
final_summary = {
    "mode": mode.mode,
    "gates": board.status_map(),
    "data_size_stages": scale_board.status_map(),
    "catalog_cells": (catalog.summary or {}).get("cell_count"),
    "catalog_donors": (catalog.summary or {}).get("donor_count"),
    "raw_cells": census.n_cells_raw,
    "approval_state": approvals.get("approval_state"),
    "decision": package.manifest["decision"],
    "artifact_paths": paths,
    "evidence_buckets": EVIDENCE_BUCKETS,
    "claim_boundary": "Separate verified-real / synthetic / metadata-only / blocked.",
}
safe_write_text(OUTPUT_ROOT / "notebook_summary.json", json.dumps(final_summary, indent=2) + "\n")
safe_write_text(OUTPUT_ROOT / "gate_board.json", json.dumps(board.to_dict(), indent=2) + "\n")
safe_write_text(OUTPUT_ROOT / "scale_board.json", json.dumps(scale_board.to_dict(), indent=2) + "\n")
print("=== G0-G9 ===")
print(gate_df.to_string(index=False))
print("=== R1-R6 ===")
print(scale_df.to_string(index=False))
print(json.dumps(final_summary, indent=2))
assert board.get("G0") is not None and board.get("G9").status == "PASS"
assert set(scale_board.status_map()) == set(["R1","R2","R3","R4","R5","R6"]) or len(scale_board.status_map()) == 6
